# Класс DenseMatrix

In [184]:
from copy import deepcopy
from typing import List, Tuple, Union, Callable, Optional # Import necessary types
import math
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
# -*- coding: utf-8 -*-

# Глобальный допуск для сравнения с нулем
_TOL = 1e-9
class DenseMatrix:
    """
    Плотная матрица без сторонних библиотек.

    Поддерживает:
      • старые имена:   n, m, shape, __getitem__, __setitem__, T …
      • новые имена:   nRows, nCols, get, set, zeros, identity …
      • арифметика, транспонирование, детерминант (Гаусс с пивотами)

    Конструктор:
      DenseMatrix(list_of_lists)            # как раньше
      DenseMatrix(rows:int, cols:int, fill=0.0)  # новая форма
    """
    # ---------- КОНСТРУКТОР ----------
    def __init__(self, data, cols=None, *, fill: float = 0.0):
        """
        data  – либо list[list[float]], либо int (число строк)
        cols  – число столбцов, если data – int
        fill  – чем заполнять при создании нулевой матрицы
        """
        if isinstance(data, int) and isinstance(cols, int):
            # Форма DenseMatrix(n, m, fill=0.0)
            if data <= 0 or cols <= 0:
                raise ValueError("Matrix dimensions must be positive")
            self._n, self._m = data, cols
            self._matrix = [[float(fill) for _ in range(self._m)]
                            for _ in range(self._n)]
        elif isinstance(data, list):
            # Форма DenseMatrix(list_of_lists)
            if not data or not isinstance(data[0], list):
                raise TypeError("Data must be a non-empty list of lists")
            self._n, self._m = len(data), len(data[0])
            if self._m == 0:
                raise ValueError("Matrix must have at least one column")
            # проверка прямоугольности и типизация
            self._matrix = []
            for r, row in enumerate(data):
                if not isinstance(row, list):
                    raise TypeError(f"Row {r} is not a list")
                if len(row) != self._m:
                    raise ValueError("Matrix must be rectangular")
                self._matrix.append([float(v) for v in row])
        else:
            raise TypeError("Invalid constructor arguments for DenseMatrix")

    # ---------- СВОЙСТВА ----------
    @property
    def n(self) -> int:            # старое имя
        return self._n
    @property
    def m(self) -> int:            # старое имя
        return self._m
    @property
    def nRows(self) -> int:        # новое имя
        return self._n
    @property
    def nCols(self) -> int:        # новое имя
        return self._m
    @property
    def shape(self) -> tuple:
        return (self._n, self._m)

    # ---------- ДОСТУП К ЭЛЕМЕНТАМ ----------
    def __getitem__(self, key: tuple) -> float:
        r, c = key
        if not (0 <= r < self._n and 0 <= c < self._m):
            raise IndexError("Index out of range")
        return self._matrix[r][c]

    def __setitem__(self, key: tuple, value):
        r, c = key
        if not (0 <= r < self._n and 0 <= c < self._m):
            raise IndexError("Index out of range")
        self._matrix[r][c] = float(value)

    # новые короткие методы - обёртки над [] / __setitem__
    def get(self, i: int, j: int) -> float:
        return self[i, j]
    def set(self, i: int, j: int, value):
        self[i, j] = value

    # ---------- ПРЕДСТАВЛЕНИЕ ----------
    def __repr__(self) -> str:
        hdr = f"DenseMatrix(shape={self.shape})"
        if self._n == 0:
            return hdr + " []"
        preview = ",\n  ".join(str(row) for row in self._matrix[:5])
        if self._n > 5:
            preview += ",\n  …"
        return hdr + "\n[\n  " + preview + "\n]"

    # ---------- КОПИРОВАНИЕ ----------
    def copy(self) -> 'DenseMatrix':
        return DenseMatrix(deepcopy(self._matrix))

    # ---------- ТРАНСПОНИРОВАНИЕ ----------
    def transpose(self) -> 'DenseMatrix':
        trans = [[self._matrix[r][c] for r in range(self._n)]
                 for c in range(self._m)]
        return DenseMatrix(trans)
    @property
    def T(self) -> 'DenseMatrix':
        return self.transpose()

    # ---------- АРИФМЕТИКА ----------
    def __add__(self, other: 'DenseMatrix') -> 'DenseMatrix':
        if not isinstance(other, DenseMatrix) or self.shape != other.shape:
            raise ValueError("Addition requires matrices of the same size")
        res = [[self._matrix[r][c] + other._matrix[r][c]
                for c in range(self._m)] for r in range(self._n)]
        return DenseMatrix(res)

    def __sub__(self, other: 'DenseMatrix') -> 'DenseMatrix':
        if not isinstance(other, DenseMatrix) or self.shape != other.shape:
            raise ValueError("Subtraction requires matrices of the same size")
        res = [[self._matrix[r][c] - other._matrix[r][c]
                for c in range(self._m)] for r in range(self._n)]
        return DenseMatrix(res)

    def __mul__(self, other):
        # Матрица * скаляр
        if isinstance(other, (int, float)):
            res = [[val * other for val in row] for row in self._matrix]
            return DenseMatrix(res)
        # Матрица * матрица
        if isinstance(other, DenseMatrix):
            if self._m != other._n:
                raise ValueError("Incompatible sizes for matrix multiplication")
            res = [[0.0] * other._m for _ in range(self._n)]
            for i in range(self._n):
                for k in range(self._m):
                    aik = self._matrix[i][k]
                    if abs(aik) < _TOL:
                        continue
                    for j in range(other._m):
                        res[i][j] += aik * other._matrix[k][j]
            return DenseMatrix(res)
        return NotImplemented

    def __rmul__(self, other):
        # Скаляр * матрица
        if isinstance(other, (int, float)):
            return self * other
        return NotImplemented

    # ---------- ОПЕРАЦИИ СОЗДАНИЯ ----------
    @staticmethod
    def zeros(n: int, m: int) -> 'DenseMatrix':
        return DenseMatrix(n, m)               # конструктор создаст нули

    @staticmethod
    def identity(n: int) -> 'DenseMatrix':
        I = DenseMatrix(n, n)
        for i in range(n):
            I._matrix[i][i] = 1.0
        return I

    # ---------- ОПРЕДЕЛИТЕЛЬ (метод Гаусса) ----------
    def det(self) -> float:
        if self._n != self._m:
            raise ValueError("Determinant only for square matrices")
        if self._n == 0:
            return 1.0
        A = deepcopy(self._matrix)
        det_val = 1.0
        swaps = 0
        for i in range(self._n):
            # поиск пивота
            pivot_row = max(range(i, self._n), key=lambda r: abs(A[r][i]))
            if abs(A[pivot_row][i]) < _TOL:
                return 0.0
            if pivot_row != i:
                A[i], A[pivot_row] = A[pivot_row], A[i]
                swaps += 1
            pivot = A[i][i]
            det_val *= pivot
            # исключение
            for r in range(i+1, self._n):
                factor = A[r][i] / pivot
                if abs(factor) < _TOL:
                    continue
                for c in range(i, self._n):
                    A[r][c] -= factor * A[i][c]
        return det_val * (-1)**swaps

# Задание 1 (Easy): Метод Гаусса для Решения СЛАУ

In [185]:
def gauss_solver(A: DenseMatrix, b: DenseMatrix) -> list:
    n = A.nRows  # dimension of system
    # Build augmented matrix [A|b] as list of lists (no external libraries)
    Aug = [[A.get(i,j) for j in range(n)] + [b.get(i,0)] for i in range(n)]

    pivot_cols = []
    row = 0
    # Forward elimination
    for col in range(n):
        # Find pivot row with nonzero A[row][col]
        pivot = None
        for r in range(row, n):
            if Aug[r][col] != 0:
                pivot = r
                break
        if pivot is None:
            continue  # no pivot in this column
        # Swap if pivot row is below current row
        if pivot != row:
            Aug[row], Aug[pivot] = Aug[pivot], Aug[row]
        # Record pivot column
        pivot_cols.append(col)
        # Normalize pivot row
        piv_val = Aug[row][col]
        Aug[row] = [val / piv_val for val in Aug[row]]
        # Eliminate variable in rows below
        for r in range(row+1, n):
            factor = Aug[r][col]
            Aug[r] = [Aug[r][c] - factor * Aug[row][c] for c in range(n+1)]
        row += 1
        if row == n:
            break

    # Detect inconsistency: 0 = nonzero in an all-zero row
    for r in range(row, n):
        if all(abs(Aug[r][c]) < 1e-12 for c in range(n)) and abs(Aug[r][n]) > 1e-12:
            raise Exception("The system is inconsistent (no solutions).")

    # Back-substitution and solution set construction
    solutions = []
    rank = row
    # Unique solution
    if rank == n:
        x = [0]*n
        # Solve from bottom pivot up
        for r in range(n-1, -1, -1):
            c = pivot_cols[r]
            # b_val minus known part
            rhs = Aug[r][n] - sum(Aug[r][j]*x[j] for j in range(c+1, n))
            x[c] = rhs
        # Wrap the solution as DenseMatrix (n×1)
        sol = DenseMatrix([[val] for val in x])
        solutions.append(sol)
    else:
        # Infinitely many solutions: find particular + basis
        # Identify free variable columns
        free_vars = [j for j in range(n) if j not in pivot_cols]
        # (1) Compute one particular solution by setting all free vars = 0
        x_part = [0]*n
        for r in range(rank-1, -1, -1):
            c = pivot_cols[r]
            rhs = Aug[r][n] - sum(Aug[r][j]*x_part[j] for j in range(c+1, n))
            x_part[c] = rhs
        sol_part = DenseMatrix([[val] for val in x_part])
        solutions.append(sol_part)
        # (2) Basis vectors for each free variable
        for fv in free_vars:
            vec = [0]*n
            vec[fv] = 1
            for r in range(rank-1, -1, -1):
                c = pivot_cols[r]
                # Compute value to cancel out term for free var = 1
                vec[c] = -sum(Aug[r][j]*vec[j] for j in range(c+1, n))
            sol_vec = DenseMatrix([[val] for val in vec])
            solutions.append(sol_vec)
    return solutions

# Задание 2 (Easy): Центрирование Данных

In [186]:
def center_data(X: 'DenseMatrix') -> 'DenseMatrix':
    if not isinstance(X, DenseMatrix):
        raise TypeError("X must be a DenseMatrix")

    n, m = X.nRows, X.nCols
    if n == 0 or m == 0:
        return X.copy()

    col_mean = [0.0] * m
    for j in range(m):
        s = 0.0
        for i in range(n):
            s += X.get(i, j)
        col_mean[j] = s / n

    centered_data = [
        [X.get(i, j) - col_mean[j] for j in range(m)]
        for i in range(n)
    ]
    return DenseMatrix(centered_data)

# Задание 3 (Easy): Вычисление Матрицы Ковариаций

In [187]:
def covariance_matrix(X_centered: 'DenseMatrix') -> 'DenseMatrix':
    if not isinstance(X_centered, DenseMatrix):
        raise TypeError("X_centered must be a DenseMatrix")

    n, m = X_centered.nRows, X_centered.nCols
    if n < 2:
        raise ValueError("At least two observations are required (n ≥ 2)")

    Xt = X_centered.T
    prod = Xt * X_centered
    norm_factor = 1.0 / (n - 1)
    C = norm_factor * prod

    return C

# Задание 4 (Normal): Нахождение Собственных Значений Методом Бисекции

In [188]:
def find_eigenvalues(C: DenseMatrix, tol: float = 1e-6) -> list:
    n = C.nRows
    diag = [C.get(i,i) for i in range(n)]
    row_sums = [sum(abs(C.get(i,j)) for j in range(n) if j != i) for i in range(n)]
    low = min(diag[i] - row_sums[i] for i in range(n))
    high = max(diag[i] + row_sums[i] for i in range(n))

    def sturm_count(x):
        prev_det = 1.0
        count = 0
        for k in range(n):
            sub = DenseMatrix(k+1, k+1)
            for i in range(k+1):
                for j in range(k+1):
                    val = C.get(i,j)
                    if i == j:
                        val -= x
                    sub.set(i, j, val)
            det_k = sub.det()
            if det_k == 0:
                det_k = 0.0
            if prev_det * det_k < 0:
                count += 1
            prev_det = det_k if det_k != 0 else prev_det
        return count

    eigenvalues = []
    lower = low
    for k in range(1, n+1):
        a, b = lower, high
        while b - a > tol:
            mid = 0.5*(a + b)
            c = sturm_count(mid)
            if c < k:
                a = mid
            else:
                b = mid
        eig = 0.5*(a + b)
        eigenvalues.append(eig)
        lower = eig
    return eigenvalues

# Задание 5 (Normal): Нахождение Собственных Векторов

In [189]:
def find_eigenvectors(C: DenseMatrix, eigenvalues: list) -> list:
    n = C.nRows
    eigenvectors = []
    I = DenseMatrix(n, n)
    for i in range(n):
        I.set(i, i, 1.0)
    for lam in eigenvalues:
        M = DenseMatrix(n, n)
        for i in range(n):
            for j in range(n):
                val = C.get(i,j)
                if i == j:
                    val -= lam
                M.set(i, j, val)
        zero_vec = DenseMatrix(n, 1)
        for i in range(n):
            zero_vec.set(i, 0, 0.0)
        sols = gauss_solver(M, zero_vec)
        for sol in sols:
            eigenvectors.append(sol)
    return eigenvectors

# Задание 6 (Normal): Вычислить долю объяснённой дисперсии

In [190]:
from typing import List

def explained_variance_ratio(eigenvalues: List[float], k: int) -> float:
    if not isinstance(eigenvalues, list) or not eigenvalues:
        raise ValueError("eigenvalues must be a non-empty list of floats")
    if not all(isinstance(val, (int, float)) for val in eigenvalues):
        raise TypeError("All eigenvalues must be numbers")
    m = len(eigenvalues)
    if not (1 <= k <= m):
        raise ValueError(f"k must be in range 1…{m}")

    # Сортируем по убыванию, если вдруг не отсортировано
    eig_sorted = sorted(eigenvalues, reverse=True)

    top_k_sum = sum(eig_sorted[:k])
    total_sum = sum(eig_sorted)

    # Защита от деления на ноль (может случиться при всех λ = 0)
    if abs(total_sum) < 1e-12:
        return 0.0

    return top_k_sum / total_sum

# Задание 7 (Hard): Реализовать полный алгоритм РСА

In [191]:
def pca(X: 'DenseMatrix', k: int) -> Tuple['DenseMatrix', float]:
    """
    Полный алгоритм PCA (Principal Component Analysis).

    Шаги
    -----
    1. Центрирование данных:        Xc = X − mean(X)
    2. Ковариационная матрица:      C  = (1/(n−1)) · Xcᵀ · Xc
    3. Собственные значения / векторы матрицы C
    4. Проекция данных:             X_proj = Xc · V_k,
       где V_k — матрица из k главных компонент (собственных векторов).

    Параметры
    ----------
    X : DenseMatrix (n × m)
        Исходная матрица данных, n — число объектов, m — число признаков.

    k : int
        Число главных компонент (1 ≤ k ≤ m).

    Возвращает
    ----------
    Tuple[DenseMatrix, float]
        • X_proj — проекция данных (n × k)
        • gamma  — доля объяснённой дисперсии для первых k компонент
    """
    # ---------- проверки ----------
    if not isinstance(X, DenseMatrix):
        raise TypeError("X must be a DenseMatrix")
    n, m = X.nRows, X.nCols
    if not (1 <= k <= m):
        raise ValueError(f"k must be in range 1…{m}")

    # ---------- 1. центрирование ----------
    Xc = center_data(X)                         # n × m

    # ---------- 2. ковариационная матрица ----------
    C = covariance_matrix(Xc)                   # m × m

    # ---------- 3. собственные значения и векторы ----------
    # 3.1 все собственные значения (получаем уже отсортированные по возр.)
    eigenvals = find_eigenvalues(C)             # длина = m
    eigenvals_desc = sorted(eigenvals, reverse=True)

    # 3.2 выбираем k наибольших и их векторы (сортировка важна!)
    top_k_vals = eigenvals_desc[:k]
    eigvecs = find_eigenvectors(C, top_k_vals)  # список DenseMatrix (m×1)

    # 3.3 нормируем векторы и формируем матрицу V_k (m × k)
    V_k = DenseMatrix(m, k)
    for col, v in enumerate(eigvecs[:k]):
        # нормировка
        norm = math.sqrt(sum(v.get(i, 0)**2 for i in range(m)))
        if norm < 1e-12:
            raise ValueError("Zero eigenvector encountered; check eigen-solver")
        for i in range(m):
            V_k.set(i, col, v.get(i, 0) / norm)

    # ---------- 4. проекция ----------
    X_proj = Xc * V_k                           # (n × m) · (m × k) → n × k

    # ---------- 5. explained variance ----------
    gamma = explained_variance_ratio(eigenvals_desc, k)

    return X_proj, gamma

In [ ]:
def plot_pca_projection(X: 'DenseMatrix',
                        k: int = 2,
                        point_size: int = 30,
                        title: Optional[str] = None,
                        show: bool = True) -> Tuple[Figure, float]:
    """
    Рисует проекцию данных, *одновременно* вызывая plt.show() / fig.show().

    Параметры
    ----------
    X : DenseMatrix
        Исходные данные (n × m).

    k : int, default=2
        Число главных компонент, используемых в PCA.

    point_size : int
        Размер маркеров в scatter-диаграмме.

    title : str | None
        Заголовок графика.

    show : bool, default=True
        Если True — сразу отображает график (fig.show()).

    Возвращает
    ----------
    (Figure, float)  → фигура и γ-доля объяснённой дисперсии.
    """
    # ---------- PCA ----------
    X_proj, gamma = pca(X, k)
    n, kk = X_proj.nRows, X_proj.nCols

    xs = [X_proj.get(i, 0) for i in range(n)]
    ys = [X_proj.get(i, 1) if kk > 1 else 0.0 for i in range(n)]

    # ---------- график ----------
    fig: Figure = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    ax.scatter(xs, ys, s=point_size, edgecolors="k")

    ax.set_xlabel("Первая главная компонента (PC1)")
    ax.set_ylabel("Вторая главная компонента (PC2)")
    ax.set_title(title or
                 f"Проекция данных (k = {k})   γ = {gamma:.3f}")
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    fig.tight_layout()

    # ----- автоматический вывод -----
    if show:
        try:
            fig.show()        # Jupyter/IPython ≥ 3.0
        except Exception:
            plt.show()        # на всякий случай

    return fig, gamma
plot_pca_projection(X, k=2)   # график появится сразу

# Задание 9 (Hard): Вычислить среднеквадратическую ошибку восстановления

In [ ]:
def reconstruction_error(X_orig: 'DenseMatrix', X_recon: 'DenseMatrix') -> float:
    """
    Вычисляет среднеквадратическую ошибку (MSE) между исходными
    и восстановленными данными.

    Формула: MSE = (1 / (n * m)) * Σ_{i,j} (X_orig[i,j] - X_recon[i,j])^2

    Вход:
        X_orig: Исходная матрица данных (n x m) типа DenseMatrix.
        X_recon: Восстановленная матрица данных (n x m) типа DenseMatrix.

    Выход:
        float: Среднеквадратическая ошибка MSE.

    Raises:
        TypeError: Если входы не являются DenseMatrix.
        ValueError: Если матрицы имеют разные размерности или пусты.
    """
    # --- Валидация входных данных ---
    if not isinstance(X_orig, DenseMatrix) or not isinstance(X_recon, DenseMatrix):
        raise TypeError("Входы X_orig и X_recon должны быть экземплярами DenseMatrix.")

    if X_orig.shape != X_recon.shape:
        raise ValueError(f"Матрицы должны иметь одинаковую размерность для вычисления ошибки: "
                         f"X_orig имеет форму {X_orig.shape}, X_recon имеет форму {X_recon.shape}")

    n = X_orig.n
    m = X_orig.m

    # Обработка случая пустых матриц
    if n == 0 or m == 0:
        # Если нет элементов для сравнения, ошибка равна 0
        return 0.0

    # --- Вычисление суммы квадратов разностей ---
    sum_sq_diff = 0.0
    for i in range(n):      # Итерация по строкам
        for j in range(m):  # Итерация по столбцам
            # Вычисляем разность элементов
            diff = X_orig[i, j] - X_recon[i, j]
            # Добавляем квадрат разности к сумме
            sum_sq_diff += diff * diff # Эквивалентно diff**2

    # --- Вычисление среднего ---
    total_elements = n * m
    mse = sum_sq_diff / total_elements

    return mse

# =============== Пример использования (для отладки) ===============
if __name__ == '__main__':
    # --- ВАЖНО: Убедитесь, что КЛАСС DenseMatrix определен ВЫШЕ ---
    # (Можно использовать упрощенный класс из предыдущих примеров для теста)
    class DenseMatrix: # Минимальная заглушка для теста
        def __init__(self, data): self._data=data; self.n=len(data); self.m=len(data[0]) if self.n>0 else 0; self.shape=(self.n,self.m)
        def __getitem__(self, key): return self._data[key[0]][key[1]]
        def __repr__(self): return f"DenseMatrix(shape={self.shape})"
    # ------------------------------------------------------------

    print("\n" + "="*20 + " Тестирование reconstruction_error " + "="*20)

    # Пример 1: Идентичные матрицы
    data_orig1 = [[1.0, 2.0], [3.0, 4.0]]
    data_recon1 = [[1.0, 2.0], [3.0, 4.0]]
    X_o1 = DenseMatrix(data_orig1)
    X_r1 = DenseMatrix(data_recon1)
    print("X_orig1:\n", X_o1)
    print("X_recon1:\n", X_r1)
    mse1 = reconstruction_error(X_o1, X_r1)
    print(f"MSE 1: {mse1:.6f} (Ожидается: 0.0)")
    assert abs(mse1 - 0.0) < 1e-9
    print("-" * 30)

    # Пример 2: Матрицы с известной разницей
    data_orig2 = [[1.0, 2.0], [3.0, 4.0]]
    data_recon2 = [[2.0, 2.0], [3.0, 5.0]] # diffs: [[-1, 0], [0, -1]]
    X_o2 = DenseMatrix(data_orig2)
    X_r2 = DenseMatrix(data_recon2)
    print("X_orig2:\n", X_o2)
    print("X_recon2:\n", X_r2)
    mse2 = reconstruction_error(X_o2, X_r2)
    # Ожидаемый расчет: ((-1)^2 + 0^2 + 0^2 + (-1)^2) / (2 * 2) = (1 + 0 + 0 + 1) / 4 = 2 / 4 = 0.5
    print(f"MSE 2: {mse2:.6f} (Ожидается: 0.5)")
    assert abs(mse2 - 0.5) < 1e-9
    print("-" * 30)

    # Пример 3: Другие значения
    data_orig3 = [[10, 0], [-1, 5]]
    data_recon3 = [[8, 1], [0, 4]] # diffs: [[2, -1], [-1, 1]]
    X_o3 = DenseMatrix(data_orig3)
    X_r3 = DenseMatrix(data_recon3)
    print("X_orig3:\n", X_o3)
    print("X_recon3:\n", X_r3)
    mse3 = reconstruction_error(X_o3, X_r3)
    # Ожидаемый расчет: (2^2 + (-1)^2 + (-1)^2 + 1^2) / 4 = (4 + 1 + 1 + 1) / 4 = 7 / 4 = 1.75
    print(f"MSE 3: {mse3:.6f} (Ожидается: 1.75)")
    assert abs(mse3 - 1.75) < 1e-9
    print("-" * 30)

    # Пример 4: Пустые матрицы
    X_o4 = DenseMatrix([])
    X_r4 = DenseMatrix([])
    print("X_orig4 (пустая):\n", X_o4)
    print("X_recon4 (пустая):\n", X_r4)
    mse4 = reconstruction_error(X_o4, X_r4)
    print(f"MSE 4: {mse4:.6f} (Ожидается: 0.0)")
    assert abs(mse4 - 0.0) < 1e-9
    print("-" * 30)

    # Пример 5: Матрицы разной формы (должна быть ошибка)
    X_o5 = DenseMatrix([[1, 2]])
    X_r5 = DenseMatrix([[1], [2]])
    print("X_orig5:\n", X_o5)
    print("X_recon5:\n", X_r5)
    try:
        mse5 = reconstruction_error(X_o5, X_r5)
        print(f"MSE 5: {mse5}") # Не должно выполниться
    except ValueError as e:
        print(f"Перехвачена ожидаемая ошибка: {e}")
    except Exception as e:
        print(f"!!! Неожиданная ошибка: {e}")
    print("-" * 30)